# Global Analysis of All Reconstructed Matrices

In [57]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [58]:
WINDOW_LENGTH = 724
STRIDE = 10
FILE_NAME = 'data_00_20'
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_LENGTH}_s{STRIDE}'
MODEL_TYPE = 'VAE'  # 'PCA', 'AE', 'linearAE', 'VAE'
RUN_NAME = 'VAE_20dim_cholesky_02'

project_root = Path.cwd().resolve().parent
model_root = project_root / 'models' / DATASET_NAME / MODEL_TYPE / RUN_NAME

use_analysis_outputs = MODEL_TYPE in {'linearAE', 'AE', 'PCA', 'VAE', 'PCA_cholesky'}
recon_dir = model_root / 'analysis_outputs' if use_analysis_outputs else model_root

DATASET_ORDER = ['test', 'train']
DATASET_FILES = {
    name: recon_dir / f'{name}_reconstructed_{RUN_NAME}.pt'
    for name in DATASET_ORDER
}
print(f'Model root: {model_root}')
RESULTS_ROOT = project_root / 'results_cholesky' / DATASET_NAME
RESULTS_DIRS = {
    name: RESULTS_ROOT / f'reconstruction_analysis_{name}' / RUN_NAME
    for name in DATASET_ORDER
}

for path in RESULTS_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

#missing = [name for name, path in DATASET_FILES.items() if not path.exists()]
#if missing:
    #missing_list = ', '.join(missing)
    #raise FileNotFoundError(f'Missing reconstructed files for: {missing_list}')

print(f'Dataset selected: {DATASET_NAME}')
print(f'Model type: {MODEL_TYPE} | Run: {RUN_NAME}')
print('Reconstructed files:')
for name, path in DATASET_FILES.items():
    print(f'  {name}: {path.name}')

Model root: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\VAE\VAE_20dim_cholesky_02
Dataset selected: data_00_20_w724_s10
Model type: VAE | Run: VAE_20dim_cholesky_02
Reconstructed files:
  test: test_reconstructed_VAE_20dim_cholesky_02.pt
  train: train_reconstructed_VAE_20dim_cholesky_02.pt


In [59]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


print('Ready to analyze reconstructed datasets:')
for name, path in DATASET_FILES.items():
    print(f'  {name}: {path.name}')

Ready to analyze reconstructed datasets:
  test: test_reconstructed_VAE_20dim_cholesky_02.pt
  train: train_reconstructed_VAE_20dim_cholesky_02.pt


# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [60]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [61]:
def compute_errors_payload(gt_corr: torch.Tensor, recon_corr: torch.Tensor):
    gt_corr_np = gt_corr.cpu().numpy()
    recon_corr_np = recon_corr.cpu().numpy()

    errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)
    errors_stats_dict = errors_stats.T.to_dict()

    return gt_corr_np, recon_corr_np, errors_df, errors_stats, errors_stats_dict

In [62]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [63]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [64]:
all_datasets_metrics = {}

for dataset_name, file_path in DATASET_FILES.items():
    print(f"\n=== Processing {dataset_name} ===")

    gt_corr, recon_corr, indices, meta = load_corr_payload(file_path)
    gt_corr_np, recon_corr_np, _, errors_stats, errors_stats_dict = compute_errors_payload(
        gt_corr,
        recon_corr,
    )

    print(f'Reconstruction error statistics ({dataset_name} dataset):')
    display(errors_stats)

    n_samples = gt_corr_np.shape[0]
    n_assets = gt_corr_np.shape[1]
    top_k_centrality = 10

    print(f"Starting MST analysis for {n_samples} matrices...")

    all_metrics = []
    for i in range(n_samples):
        curr_gt = gt_corr_np[i]
        curr_recon = recon_corr_np[i]

        mst_gt = corr_to_mst(curr_gt)
        mst_recon = corr_to_mst(curr_recon)

        metrics = compare_mst_metrics(
            mst_gt,
            mst_recon,
            n_assets=n_assets,
            top_k=top_k_centrality,
        )

        all_metrics.append(metrics)

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{n_samples} matrices...")

    results_df = pd.DataFrame(all_metrics)
    stats_df = results_df.describe().T
    stats_dict = stats_df.to_dict(orient='index')

    combined_metrics = {
        'reconstruction_errors': errors_stats_dict,
        'mst_metrics': stats_dict,
    }

    output_path = RESULTS_DIRS[dataset_name] / f'reconstruction_errors_{dataset_name}.json'
    with open(output_path, 'w') as f:
        json.dump(combined_metrics, f, indent=4)

    print(f"Saved metrics to: {output_path}")
    print(f"\n--- Summary Statistics ({dataset_name} dataset) ---")
    display(stats_df)

    all_datasets_metrics[dataset_name] = {
        'errors_stats': errors_stats,
        'mst_stats': stats_df,
    }


=== Processing test ===
Reconstruction error statistics (test dataset):


,mean,std,min,median,max
MSE,0.000074,0.000084,0.000003,0.000064,0.000501
MAE,0.005170,0.002760,0.001251,0.005206,0.016983
Frobenius,2.767125,1.426083,0.649115,2.885143,8.101620


Starting MST analysis for 42 matrices...
Processed 10/42 matrices...
Processed 20/42 matrices...
Processed 30/42 matrices...
Processed 40/42 matrices...
Saved metrics to: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results_cholesky\data_00_20_w724_s10\reconstruction_analysis_test\VAE_20dim_cholesky_02\reconstruction_errors_test.json

--- Summary Statistics (test dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,42.0,91.010421,4.514637,74.515235,88.711911,91.274238,94.459834,96.675900
edge_jaccard_pct,42.0,83.798127,7.294268,59.381898,79.713961,83.951731,89.501312,93.565684
degree_l1,42.0,0.070508,0.021128,0.038674,0.055249,0.066298,0.082873,0.121547
avg_path_len,42.0,7.584592,1.051163,5.598302,7.218289,7.517954,8.213715,10.232269
avg_path_len_recon,42.0,7.500550,1.045439,5.624142,7.138004,7.438137,7.990102,10.196994
avg_path_len_diff,42.0,0.254131,0.251939,0.014834,0.062500,0.163825,0.339803,0.985241
avg_path_len_unweighted,42.0,10.062628,1.365729,7.987114,8.971002,9.915650,10.959994,12.760426
avg_path_len_unweighted_recon,42.0,9.946813,1.372953,8.075068,8.852883,9.821062,10.659498,12.933839
avg_path_len_unweighted_diff,42.0,0.394897,0.379492,0.027196,0.126639,0.305099,0.552433,1.482653
betweenness_topk_overlap_pct,42.0,83.095238,16.303239,40.000000,80.000000,90.000000,97.500000,100.000000



=== Processing train ===
Reconstruction error statistics (train dataset):


,mean,std,min,median,max
MSE,0.000004,0.000007,1.095684e-07,0.000003,0.000089
MAE,0.001186,0.000439,2.504673e-04,0.001182,0.002841
Frobenius,0.646108,0.350574,1.198260e-01,0.609521,3.409915


Starting MST analysis for 288 matrices...
Processed 10/288 matrices...
Processed 20/288 matrices...
Processed 30/288 matrices...
Processed 40/288 matrices...
Processed 50/288 matrices...
Processed 60/288 matrices...
Processed 70/288 matrices...
Processed 80/288 matrices...
Processed 90/288 matrices...
Processed 100/288 matrices...
Processed 110/288 matrices...
Processed 120/288 matrices...
Processed 130/288 matrices...
Processed 140/288 matrices...
Processed 150/288 matrices...
Processed 160/288 matrices...
Processed 170/288 matrices...
Processed 180/288 matrices...
Processed 190/288 matrices...
Processed 200/288 matrices...
Processed 210/288 matrices...
Processed 220/288 matrices...
Processed 230/288 matrices...
Processed 240/288 matrices...
Processed 250/288 matrices...
Processed 260/288 matrices...
Processed 270/288 matrices...
Processed 280/288 matrices...
Saved metrics to: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results_cholesky\data_00_20_w724_s10\

,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,288.0,97.936865,1.019463,94.736842,97.506925,98.060942,98.614958,100.000000
edge_jaccard_pct,288.0,95.976552,1.950345,90.000000,95.135135,96.195652,97.267760,100.000000
degree_l1,288.0,0.037638,0.015742,0.000000,0.027624,0.033149,0.045580,0.077348
avg_path_len,288.0,7.853456,1.417111,5.000791,6.869513,7.625463,8.919166,11.573367
avg_path_len_recon,288.0,7.868956,1.432883,4.956724,6.850160,7.626213,8.865147,11.592046
avg_path_len_diff,288.0,0.126924,0.195885,0.000206,0.017784,0.050595,0.136979,1.346679
avg_path_len_unweighted,288.0,10.371716,1.533951,7.142713,9.262565,10.110421,11.585964,14.683491
avg_path_len_unweighted_recon,288.0,10.396531,1.558963,7.083179,9.320748,10.076476,11.690103,14.702729
avg_path_len_unweighted_diff,288.0,0.189537,0.306696,0.000000,0.024299,0.069176,0.189395,2.237370
betweenness_topk_overlap_pct,288.0,94.201389,9.915377,30.000000,90.000000,100.000000,100.000000,100.000000
